# Land of 10,000 Lakes? Let's count. 🛶

**A SciPy lightning talk running entirely in this browser tab.** This site is
[JupyterLite](https://jupyterlite.readthedocs.io/): CPython compiled to
WebAssembly ([Pyodide](https://pyodide.org)), kernel in a Web Worker, no
server anywhere.

Minnesota's license plates say *10,000 lakes*. The DNR's classic count is
**11,842 lakes of 10+ acres** — and the current [DNR Hydrography
inventory](https://gisdata.mn.gov/dataset/water-dnr-hydrography) delineates
even more. We'll load every numbered lake basin in the state and map **lake
density** with a Gaussian KDE: a *billion* kernel evaluations — too much
for one interpreter.

WebAssembly Python can't `fork` and can't spawn threads, so we'll do what
`multiprocessing` would do: **more interpreters.** A pool of nested Web
Workers, each hosting its own Pyodide, driven by a stock `dask` graph —
first without the pool, then with it.

In [ ]:
%pip install -q pyodide-pool dask

The `pyodide-pool` wheel ships with this site (a local piplite index);
`dask` comes from PyPI. Now the lakes — one CSV row per DNR-numbered lake
basin of 10+ acres, extracted from the state's authoritative hydrography
data (`scripts/make-mn-lakes-data.py` in the repo).

In [ ]:
import numpy as np
from js import location
from pyodide.http import pyfetch

# Site base URL — the kernel worker lives under {site}/extensions/…, so this
# works at a domain root and under a GitHub Pages project subpath alike.
BASE_URL = location.href.split("extensions/")[0]

lines = (await (await pyfetch(BASE_URL + "files/data/mn_lakes.csv")).string()).splitlines()
lat, lon, acres = np.loadtxt(
    lines[1:], delimiter=",", usecols=(1, 2, 3), comments=None, unpack=True
)
biggest = [line.split(",")[0] for line in lines[1:6]]

print(f"The plates say 10,000 · the classic DNR count says 11,842")
print(f"Today's DNR Hydrography: {len(lat):,} lake basins of 10+ acres")
print("Biggest:", " · ".join(biggest))

## The workload: a lake-density map

Project every lake to km, lay a grid over the state, and evaluate a
Gaussian KDE — **lakes per km²** — at every grid cell. Pure NumPy, embarrassingly
parallel across grid rows: each `dask.delayed` task computes one horizontal
band of the map, and `np.vstack` stitches them together.

In [ ]:
import dask

R = 6371.0  # km
lat0 = np.deg2rad(lat.mean())
px = (R * np.cos(lat0) * np.deg2rad(lon)).astype(np.float32)  # km east
py = (R * np.deg2rad(lat)).astype(np.float32)                 # km north

GRID_W, GRID_H, TASKS = 240, 320, 12
PAD = 25.0
gx = np.linspace(px.min() - PAD, px.max() + PAD, GRID_W, dtype=np.float32)
gy = np.linspace(py.min() - PAD, py.max() + PAD, GRID_H, dtype=np.float32)
bw = np.float32(len(px) ** (-1 / 6) * np.mean([px.std(), py.std()]))  # Scott's rule


def kde_band(band_y, grid_x, x, y, bw, block=512):
    """Gaussian KDE (lakes per km²) over one horizontal band of the grid."""
    import numpy as np

    dens = np.zeros((band_y.size, grid_x.size), dtype=np.float32)
    inv = np.float32(-0.5) / (bw * bw)
    gxx, gyy = grid_x[None, :, None], band_y[:, None, None]
    for i in range(0, x.size, block):  # cache-sized point blocks
        dx = gxx - x[i : i + block]
        dy = gyy - y[i : i + block]
        dens += np.exp(inv * (dx * dx + dy * dy)).sum(axis=-1)
    return dens / (2 * np.pi * bw * bw)


parts = [dask.delayed(kde_band)(band, gx, px, py, bw) for band in np.array_split(gy, TASKS)]
density_graph = dask.delayed(np.vstack)(parts)
print(f"{TASKS} tasks · {GRID_W * GRID_H * len(px) / 1e9:.2f} billion kernel evaluations")

## Without the pool…

`dask`'s own single-threaded scheduler, right here in the kernel — one
WebAssembly interpreter doing all the work.

In [ ]:
import time

t0 = time.perf_counter()
density_serial = density_graph.compute(scheduler="synchronous")
serial_s = time.perf_counter() - t0
print(f"1 interpreter: {serial_s:.1f} s")

## …and with it

`create_pool` imports the pool's JS bundle from inside the kernel and spawns
**nested Web Workers**, each booting its own Pyodide — in parallel. The first
task each worker sees also **mirrors the kernel's packages** (NumPy, dask)
into it: pay once, reuse for every later task.

In [ ]:
import asyncio
import pyodide_pool

POOL_SIZE = 4
t0 = time.perf_counter()
pool = await pyodide_pool.create_pool(
    pool_size=POOL_SIZE, js_url=BASE_URL + "files/assets/pyodide-pool.browser.js"
)
await asyncio.gather(*(pool.submit(lambda i=i: i) for i in range(POOL_SIZE)))
print(f"{POOL_SIZE} workers booted + warmed in {time.perf_counter() - t0:.1f} s")

In [ ]:
t0 = time.perf_counter()
density = await pyodide_pool.compute(density_graph)  # same graph, async scheduler
pool_s = time.perf_counter() - t0
speedup = serial_s / pool_s

assert np.allclose(density, density_serial, rtol=1e-4)
print(f"{POOL_SIZE} interpreters: {pool_s:.1f} s — {speedup:.1f}× faster, same map")

## See it

(The first `matplotlib` import downloads it into the kernel — a few seconds.
It stays out of the workers: they only ever needed NumPy and dask.)

In [ ]:
import matplotlib.pyplot as plt

outline = np.genfromtxt(
    (await (await pyfetch(BASE_URL + "files/data/mn_outline.csv")).string()).splitlines(),
    delimiter=",", skip_header=1,
)
ox = R * np.cos(lat0) * np.deg2rad(outline[:, 0])
oy = R * np.deg2rad(outline[:, 1])

fig, ax = plt.subplots(figsize=(5.4, 6.2))
ax.plot(ox, oy, color="#44403c", linewidth=1.1, zorder=3)
ax.scatter(px, py, s=np.clip(acres / 300, 0.15, 110), c="#2a78d6", alpha=0.35, linewidths=0)
for i, name in enumerate(biggest):
    ax.annotate(name, (px[i], py[i]), fontsize=7, color="#1c1917",
                xytext=(4, 3), textcoords="offset points")
ax.set_title(f"the inputs: {len(px):,} lakes ≥ 10 acres", fontsize=11)
ax.set_aspect("equal")
ax.set_axis_off()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.2))
mesh = ax.pcolormesh(gx, gy, density * 1000, cmap="viridis", shading="auto")
ax.plot(ox, oy, color="white", linewidth=1.0, alpha=0.9)
iy, ix = np.unravel_index(density.argmax(), density.shape)
ax.plot(gx[ix], gy[iy], "w*", markersize=13, markeredgecolor="black")
ax.annotate(f"peak: {density.max() * 1000:,.0f} lakes / 1000 km²",
            (gx[ix], gy[iy]), xytext=(10, -2), textcoords="offset points",
            color="white", fontsize=9, fontweight="bold")
fig.colorbar(mesh, ax=ax, label="lakes per 1,000 km²", shrink=0.75)
ax.set_title("the result: Minnesota lake density (Gaussian KDE)", fontsize=11)
ax.set_aspect("equal")
ax.set_axis_off()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 2.1))
labels = ["1 interpreter", f"{POOL_SIZE} interpreters"]
bars = ax.barh(labels, [serial_s, pool_s], color=["#b3b2ab", "#2a78d6"], height=0.6)
ax.bar_label(bars, [f" {serial_s:.1f} s", f" {pool_s:.1f} s  ({speedup:.1f}× faster)"])
ax.invert_yaxis()
ax.set_xlim(0, serial_s * 1.35)
ax.set_xlabel("same dask graph, same browser tab")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## How: `multiprocessing`, translated to the browser

The philosophy is exactly `multiprocessing`'s — **N interpreters ×
message passing** (cloudpickle) — because one WASM CPython has no `fork`,
no threads. What the browser changes:

- **The event loop is load-bearing.** The kernel shares its thread with the
  pool's `postMessage` plumbing, so a blocking scheduler would deadlock the
  page. The dask scheduler is **async** (`await pyodide_pool.compute(...)`)
  and keeps up to `pool_size` ready tasks in flight; truly *blocking* calls
  exist only where JSPI stack-switching makes them safe — that's the
  `wasm_multiprocessing` shim (`from wasm_multiprocessing import Pool`, a
  one-line port of stdlib code; see the other notebooks on this site).
- **No shared filesystem, no pre-built env.** Workers are stateless: every
  task rides with a snapshot of the kernel's installed packages, replayed
  idempotently on arrival — the first task pays the install, the rest skip it.

## Why it's fast

- **Warm load** — the pool boots all interpreters in parallel (~1–2 s each)
  and recycles them LIFO via
  [`@fideus-labs/worker-pool`](https://www.npmjs.com/package/@fideus-labs/worker-pool),
  so tasks always land on the warmest worker and never re-boot.
- **Message efficiency** — calls travel as cloudpickled bytes in
  **zero-copy transferables**: ~0.7 ms per task + ~10 ms/MiB of payload.
  This whole map moved ≈1 MB of float32 for its speedup.
- **Honest ceiling** — near-linear only on chunky CPU-bound tasks like
  these bands; per-message overhead makes tiny tasks pointless. Measured
  across Node and browsers: ~3× on 4 workers for pure-Python workloads.

*Everything here is open: the JS `PyodidePool`, the async dask scheduler,
and the `multiprocessing` shim, with design docs and benchmark reports, in
the repository this site is built from — `demos/jupyterlite/` has these
notebooks.*

In [ ]:
# Install it!
%pip install pyodide_pool
%pip install wasm_multiprocessing

## Success marker

In [ ]:
assert speedup > 1.0
print(f"SCIPY_DEMO_OK {speedup:.2f}")